In [ ]:
!pip install rasterio

In [ ]:
# ============================================
# IMPORTS
# ============================================

import os
import rasterio
import numpy as np

from google.colab import drive

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.models import Sequential

from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

# ============================================
# MOUNT GOOGLE DRIVE
# ============================================

drive.mount('/content/drive')

# ============================================
# DATASET PATH
# ============================================

DATASET_PATH = "/content/drive/MyDrive/Climate_Project/dataset/rainfall"

# ============================================
# LOAD TIFF FILES
# ============================================

files = sorted([

    f for f in os.listdir(DATASET_PATH)

    if f.endswith(".tif")
])

print("Total TIFF Files:", len(files))

# ============================================
# FEATURE EXTRACTION
# ============================================

features = []

for file in files:

    path = os.path.join(
        DATASET_PATH,
        file
    )

    print("Processing:", file)

    with rasterio.open(path) as src:

        image = src.read(1)

        # REMOVE NaN VALUES
        image = np.nan_to_num(image)

        # ====================================
        # EXTRACT FEATURES
        # ====================================

        mean_val = np.mean(image)

        max_val = np.max(image)

        min_val = np.min(image)

        std_val = np.std(image)

        # STORE FEATURES

        features.append([

            mean_val,
            max_val,
            min_val,
            std_val
        ])

# ============================================
# CONVERT TO NUMPY ARRAY
# ============================================

features = np.array(features)

print("\nFeature Shape:", features.shape)

# ============================================
# NORMALIZATION
# ============================================

scaler = MinMaxScaler()

features_scaled = scaler.fit_transform(
    features
)

# ============================================
# CREATE SEQUENCES
# ============================================

SEQUENCE_LENGTH = 6

X = []
y = []

for i in range(

    SEQUENCE_LENGTH,

    len(features_scaled)

):

    X.append(

        features_scaled[
            i-SEQUENCE_LENGTH:i
        ]
    )

    # TARGET = NEXT MONTH MEAN VALUE

    y.append(
        features_scaled[i][0]
    )

# ============================================
# CONVERT TO ARRAYS
# ============================================

X = np.array(X)

y = np.array(y)

print("\nX Shape:", X.shape)

print("y Shape:", y.shape)

# ============================================
# TRAIN TEST SPLIT
# ============================================

train_size = int(
    len(X) * 0.8
)

X_train = X[:train_size]

X_test = X[train_size:]

y_train = y[:train_size]

y_test = y[train_size:]

print("\nTraining Samples:", len(X_train))

print("Testing Samples:", len(X_test))

# ============================================
# BUILD LSTM MODEL
# ============================================

model = Sequential()

# FIRST LSTM LAYER

model.add(

    LSTM(

        64,

        return_sequences=True,

        input_shape=(

            X_train.shape[1],

            X_train.shape[2]
        )
    )
)

model.add(
    Dropout(0.2)
)

# SECOND LSTM

model.add(
    LSTM(64)
)

model.add(
    Dropout(0.2)
)

# OUTPUT LAYER

model.add(
    Dense(1)
)

# ============================================
# COMPILE MODEL
# ============================================

model.compile(

    optimizer='adam',

    loss='mse',

    metrics=['mae']
)

# ============================================
# MODEL SUMMARY
# ============================================

model.summary()

# ============================================
# TRAIN MODEL
# ============================================

history = model.fit(

    X_train,

    y_train,

    epochs=70,

    batch_size=8,

    validation_data=(

        X_test,

        y_test
    )
)

# ============================================
# PREDICTIONS
# ============================================

predictions = model.predict(X_test)

# ============================================
# EVALUATION
# ============================================

r2 = r2_score(
    y_test,
    predictions
)

mae = mean_absolute_error(
    y_test,
    predictions
)

rmse = np.sqrt(

    mean_squared_error(
        y_test,
        predictions
    )
)

# ============================================
# PRINT RESULTS
# ============================================

print("\n==========================")

print("MODEL RESULTS")

print("==========================")

print(f"R2 Score: {r2}")

print(f"MAE: {mae}")

print(f"RMSE: {rmse}")

print("==========================")

# ============================================
# SAVE MODEL
# ============================================

model.save(

    "/content/drive/MyDrive/climate_lstm_model.keras"
)

print("\nMODEL SAVED SUCCESSFULLY")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total TIFF Files: 44
Processing: Rainfall_2020_1.tif
Processing: Rainfall_2020_2.tif
Processing: Rainfall_2020_3.tif
Processing: Rainfall_2020_4.tif
Processing: Rainfall_2020_5.tif
Processing: Rainfall_2020_6.tif
Processing: Rainfall_2020_7.tif
Processing: Rainfall_2020_8.tif
Processing: Rainfall_2020_9.tif
Processing: Rainfall_2021_10.tif
Processing: Rainfall_2021_11.tif
Processing: Rainfall_2021_12.tif
Processing: Rainfall_2021_8.tif
Processing: Rainfall_2021_9.tif
Processing: Rainfall_2022_1.tif
Processing: Rainfall_2022_10.tif
Processing: Rainfall_2022_11.tif
Processing: Rainfall_2022_12.tif
Processing: Rainfall_2022_2.tif
Processing: Rainfall_2022_3.tif
Processing: Rainfall_2022_4.tif
Processing: Rainfall_2022_5.tif
Processing: Rainfall_2022_6.tif
Processing: Rainfall_2022_7.tif
Processing: Rainfall_2022_8.tif
Processing: Rainfall_2022_9.tif
Processing: 

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 6, 64)          │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 50,753 (198.25 KB)

 Trainable params: 50,753 (198.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 5s 182ms/step - loss: 0.2587 - mae: 0.3785 - val_loss: 0.1831 - val_mae: 0.3155
Epoch 2/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.1849 - mae: 0.3277 - val_loss: 0.1681 - val_mae: 0.3499
Epoch 3/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.1709 - mae: 0.3569 - val_loss: 0.1840 - val_mae: 0.3848
Epoch 4/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.1795 - mae: 0.3739 - val_loss: 0.1758 - val_mae: 0.3759
Epoch 5/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - loss: 0.1536 - mae: 0.3474 - val_loss: 0.1567 - val_mae: 0.3469
Epoch 6/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - loss: 0.1529 - mae: 0.3414 - val_loss: 0.1458 - val_mae: 0.3278
Epoch 7/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - loss: 0.1437 - mae: 0.3267 - val_loss: 0.1387 - val_mae: 0.3175
Epoch 8/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss: 0.1423 - mae: 0.3170 - val_loss: 0.1330 - val_mae: 0.3106
Epoch 9/50
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - loss: 0.1377 - mae: 0.3182 -

In [9]:
model.save(
    "/content/drive/MyDrive/Climate_Project/model/climate_lstm_model.keras"
)

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/Climate_Project/model/climate_lstm_model.keras'

In [10]:
import os

# CREATE MODEL DIRECTORY
os.makedirs(
    "/content/drive/MyDrive/Climate_Project/model",
    exist_ok=True
)

# SAVE MODEL
model.save(
    "/content/drive/MyDrive/Climate_Project/model/climate_lstm_model.keras"
)

print("MODEL SAVED SUCCESSFULLY")

MODEL SAVED SUCCESSFULLY
